# 环境搭建与第一个 PyPTO Pro Kernel

本节完成两件事：确认开发环境就绪；跑通一个完整的向量加法算子——从定义 Kernel、准备数据、启动执行到精度验证的完整闭环。这个闭环是后续所有算子开发的模板。


## 1. 环境要求

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">项目</th><th style="text-align:left;">要求</th></tr>
<tr><td>支持硬件</td><td>Ascend 950PR / 950DT 系列产品</td></tr>
<tr><td>CANN</td><td>9.2.0 及以上（使用 CANN 包内置 PyPTO，无需单独安装 pypto 包）</td></tr>
<tr><td>Python</td><td>3.11 / 3.12</td></tr>
<tr><td>依赖</td><td>torch、torch_npu（用于构造输入与结果验证）</td></tr>
</table>

> 目标是在 CANNLab 的 950 环境验证；目前尚未登记已验证镜像与内核。具备对应环境后，在章节目录打开 Notebook，确保相对 answer/images 路径有效。运行前需加载 CANN 环境变量（`source /usr/local/Ascend/ascend-toolkit/set_env.sh`，具体路径以实际安装为准）。


## 2. 环境自检

先配置两个关键环境变量并完成导入自检：`TILE_FWK_DEVICE_ID` 指定运行的 NPU 设备号；`TORCH_DEVICE_BACKEND_AUTOLOAD` 关闭设备后端自动加载（避免非目标设备的后端加载干扰）。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('PyPTO Pro 环境导入成功')

看到“导入成功”只说明 Python 包可以导入；设备、驱动、编译器与运行时是否可用，还需下面的 Kernel 在目标 950 环境实际启动并通过校验。CANN 环境变量应在启动 Jupyter 的终端中加载，使 Notebook 内核继承；在单独的 `!source` 子 shell 中执行不能改变已有内核的环境。


## 3. 算子规格

我们要实现的算子非常简单，先用一张规格表把它定义清楚——**先明确规格，再写代码**，这是贯穿全课程的算子开发习惯：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">项目</th><th style="text-align:left;">说明</th></tr>
<tr><td>数学表达式</td><td><code>out = x + y</code></td></tr>
<tr><td>输入</td><td>x、y：Shape [8, 256]，FP32，ND</td></tr>
<tr><td>输出</td><td>out：Shape [8, 256]，FP32，ND</td></tr>
<tr><td>使用的主要接口</td><td>pl.TileType、pl.make_tile_group、pl.load、pl.add、pl.store</td></tr>
</table>


## 4. 算子分析三步法

拿到一个算子需求，先走三步分析，再动手写代码——这是贯穿全课程（以及 Ascend C 体系）的方法论：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">步骤</th><th style="text-align:left;">本例的答案</th></tr>
<tr><td><b>① 明确数学表达式与计算逻辑</b></td><td><code>out = x + y</code>，逐元素相加，无归约、无状态</td></tr>
<tr><td><b>② 明确输入输出</b></td><td>x、y、out 各 [8, 256] FP32 ND（即上文的规格表）</td></tr>
<tr><td><b>③ 确定核函数与参数</b></td><td>一个 Kernel；三个 Tensor 参数（类型标注即规格）</td></tr>
</table>

三步的产出直接映射到代码结构：① 决定 Kernel 内计算段的写法，② 决定 dtype/shape 与精度校验方式，③ 决定函数签名。后续每个实战章节都会先走这三步。


## 5. 完整代码

下面的代码单元包含完整闭环：环境变量、Kernel 定义、输入构造、启动与验证。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

ROWS = 8
COLS = 256


@pl.jit(auto_mutex=True)
def add_kernel(
    x: pl.Tensor[[ROWS, COLS], pl.DT_FP32],
    y: pl.Tensor[[ROWS, COLS], pl.DT_FP32],
    out: pl.Tensor[[ROWS, COLS], pl.DT_FP32],
):
    tile_type = pl.TileType(
        shape=[ROWS, COLS],
        dtype=pl.DT_FP32,
        target_memory=pl.MemorySpace.Vec,
    )

    x_group = pl.make_tile_group(type=tile_type, addrs=0x0000, mutex_ids=[0])
    y_group = pl.make_tile_group(type=tile_type, addrs=0x2000, mutex_ids=[1])
    out_group = pl.make_tile_group(type=tile_type, addrs=0x4000, mutex_ids=[2])

    with pl.section_vector():
        x_tile = x_group.current()
        y_tile = y_group.current()
        out_tile = out_group.current()

        # 1. GM -> UB
        pl.load(x_tile, x, [0, 0])
        pl.load(y_tile, y, [0, 0])

        # 2. 逐元素相加（SIMD：一次接口调用处理整个 Tile）
        pl.add(out_tile, x_tile, y_tile)

        # 3. UB -> GM
        pl.store(out, out_tile, [0, 0])

In [ ]:
device_id = int(os.environ.get('TILE_FWK_DEVICE_ID', 0))
device = f'npu:{device_id}'
torch.npu.set_device(device)

torch.manual_seed(0)

x = torch.randn(ROWS, COLS, dtype=torch.float32, device=device)
y = torch.randn(ROWS, COLS, dtype=torch.float32, device=device)
out = torch.full_like(x, float('nan'))

# 首次调用触发 JIT 编译并执行；编译签名相同时复用编译结果
add_kernel[None, 1](x, y, out)  # 本例没有多核切分，显式只启动一个 Vector 核

torch.npu.synchronize()

golden = torch.add(x, y)

assert torch.isfinite(out).all().item(), '输出存在未写回位置或非有限值'
torch.testing.assert_close(out, golden, rtol=1e-5, atol=1e-6)

print('向量加法 Kernel 通过精度校验！')
print('PyPTO Pro 输出（前 8 个元素）:', out.flatten()[:8].tolist())
print('PyTorch 参考输出（前 8 个元素）:', golden.flatten()[:8].tolist())

**【待 950 验证】** 预期输出：`向量加法 Kernel 通过精度校验！`，且 PyPTO Pro 输出与 PyTorch 参考输出的前 8 个元素逐项一致（浮点误差在阈值内）。


## 6. 代码逐段解读

本节解读点到为止——先建立"跑通"的体感；Tile、auto_mutex、多缓冲等概念的系统解释在第 2 章（SIMD 编程模型）进行。

**Kernel 定义**（对照 01.04 的编译器视角与 01.03 的硬件地图阅读）：

1. `@pl.jit(auto_mutex=True)`：声明这是一个 JIT Kernel；`auto_mutex=True` 表示由 Pro 自动管理搬入、计算、搬出之间的流水依赖（01.03 的"异步流水与同步"），本例的核内数据依赖无需手动同步；跨核依赖另见 02.06；
2. `x: pl.Tensor[[ROWS, COLS], pl.DT_FP32]`：类型标注即参数声明——`Tensor` 描述 GM 中的多维数据，中括号内是静态 Shape，`DT_FP32` 是数据类型；
3. `pl.TileType(shape=..., dtype=..., target_memory=pl.MemorySpace.Vec)`：定义片上 Tile 的形状、类型与目标存储（`MemorySpace.Vec` 即 AIV 侧 UB）；
4. `pl.make_tile_group(type=..., addrs=0x..., mutex_ids=[n])`：在 UB 上为每个 Tile 分配片上地址。**不同 Tile Group 必须使用互不重叠的地址**——本例三个 Tile 各占 8×256×4 字节 = 8KB，地址分别取 0x0000 / 0x2000 / 0x4000；
5. `with pl.section_vector():`：Vector 执行域的代码段，数据搬运与计算在此组织；
6. `group.current()`：取当前 Tile Group 的活动 Tile（不推进轮转位置；多缓冲用 `next()` 推进，第 2 章展开）；
7. `pl.load(tile, tensor, [0, 0])` / `pl.store(tensor, tile, [0, 0])`：GM 与 UB 之间的搬运，`[0, 0]` 是 Tensor 上的起始坐标；
8. `pl.add(out_tile, x_tile, y_tile)`：SIMD 逐元素加——**一次接口调用处理整个 Tile**，底层可能展开成多条指令，不能按一次 API 调用推断指令条数。

**Host 侧**：用 `torch.randn(..., device=device)` 准备 NPU 上的输入；`add_kernel[None, 1](x, y, out)` 在当前流启动单核，首次调用触发 JIT 编译并异步执行；`torch.npu.synchronize()` 等待 Device 完成；最后与 `torch.add` 的结果做 `assert_close` 精度校验。


## 7. SIMT 写法对照

同一个向量加法，用 SIMT 写是什么样？对比着看，你能直观感受 01.05 的两种范式：

In [ ]:
@pl.vector_function(mode='simt', max_threads=COLS)
def add_by_thread(x_tile, y_tile, out_tile):
    tid = pl.simt.linear_thread_idx()
    for row in pl.range(ROWS):
        out_tile[row, tid] = x_tile[row, tid] + y_tile[row, tid]


# 在外层 Kernel 的 section_vector 中替换 SIMD 计算段即可启动：
#     add_by_thread[COLS](x_tile, y_tile, out_tile)
# 方括号内为线程数：256 线程，每线程处理同一列的 ROWS 个元素
# 本单元仅定义对照函数；替换计算段后需重新定义外层 Kernel、启动并校验全部输出

对照要点：SIMD 版一行 `pl.add(out_tile, x_tile, y_tile)` 完成整个 Tile 的相加；SIMT 版由 256 个线程各自处理第 `tid` 列的 8 行，覆盖完整的 `[8,256]` Tile。只写 `[0, tid]` 会漏掉其余 7 行。对这种规整算子，SIMD 表达更简洁，性能差异仍需实测；但若每个元素的加法系数要从一张查找表里离散取，SIMT 的写法就能自然表达。SIMT 的完整语法（线程层次、块内同步、原子操作）在第 6~7 章展开。


## 8. 常见问题

- 报错 `maybe others are using current device`：用 `npu-smi info` 查看设备占用，关闭多余 notebook 或清理残留进程；
- 首次调用明显慢于后续：正常现象——首次调用执行"绑定→解析→IR→代码生成→编译加载"全流程，后续按编译签名复用（01.04）。


### 规格变更时，先更新这张账

原例 [8,256] FP32 每块为 0x2000 B，三块共 24 KiB。若换成 [16,512] FP32，每块变成 0x8000 B，地址可取 0、0x8000、0x10000；若同时换成 FP16，则每块为 0x4000 B，可取 0、0x4000、0x8000。只改 Tensor shape 而保留原地址，会让多个 Tile 覆盖同一段 UB。

按顺序检查参数标注、TileType、Host 输入 shape/dtype、字节预算与 Golden 容差。输出初始化为 NaN，检查所有元素有限，再做全量 `assert_close`；打印前 8 个元素只能帮助观察，无法证明后 2040 个元素已正确写出。

本例没有核间切分，始终显式启动一个核。把启动数改成 2，并不会自动让两个核各处理一半。


## 9. 课后练习

1. （填空）`make_tile_group` 的 `addrs` 参数为 Tile 分配片上地址，多个 Tile Group 的地址必须 ______。
2. （选择）`auto_mutex=True` 的作用是：
   A. 自动管理搬入/计算/搬出之间的流水依赖<br/>
   B. 自动选择核数<br/>
   C. 自动做精度校验
3. （实践）把 Shape 从 [8, 256] 改为 [16, 512]，思考并验证：哪些地方需要同步修改？Tile 地址分配是否仍然安全？

In [ ]:
!cat ./answer/01.06_answer.txt

## 10. 本节小结

- 完整闭环 = **规格表 → Kernel 定义（jit + 类型标注 + Tile 规划 + section_vector）→ Host 构造输入 → 启动与同步 → 精度校验**；
- `make_tile_group` 显式规划片上地址，地址不可重叠；
- SIMD 一行接口处理整个 Tile，SIMT 由线程逐元素处理——同一算子、两种表达；
- 首次调用慢是 JIT 编译的正常行为。

至此第 1 章的概念与首次实操全部完成。下一节通过实践检验本章所学。